# Oturum 6 — `bentopy` ile Kalabalık Hücresel Sistemler

**Biyofizik 2026 Kursu · Dr. Öğr. Üyesi Ekrem Yaşar**

Kurs boyunca izlenen ölçek gelişimi:

| Oturum | Sistem | Karakteristik uzunluk |
|---|---|---|
| 2 | Çözelti içinde tek lizozim molekülü | yaklaşık 5 nm |
| 3 | Membranda tek reseptör, atomistik | yaklaşık 10 nm |
| 4 | Membranda tek reseptör, kaba-taneli | yaklaşık 12 nm |
| 6 | Kalabalık membran ve sitozol | 40–100 nm |

Kursun başlığında yer alan "büyük ve kalabalık hücresel sistemler" ifadesi bu
oturumun konusunu oluşturmaktadır.


---
## Kalabalık ortam koşullarının önemi

Moleküler simülasyonların büyük bölümü proteinleri seyreltik çözelti
koşullarında incelemektedir. Hücre içi ortam bu varsayımdan belirgin biçimde
ayrılmaktadır:

- Sitoplazmada toplam makromolekül derişimi yaklaşık 300 g/L düzeyindedir;
  hacmin %20–30'u makromoleküller tarafından işgal edilmektedir.
- Biyolojik membranlarda protein/lipit oranı yüksektir.
- Kalabalık koşulları difüzyonu yavaşlatmakta ve bağlanma dengelerini
  kaydırmaktadır.

**Yöntemsel güçlük.** Çok sayıda makromolekülün çakışma oluşturmadan, uygun
yönelimlerle ve hedeflenen derişimde bir simülasyon kutusuna yerleştirilmesi
elle yapılabilecek bir işlem değildir.

**Çözüm.** [`bentopy`](https://github.com/marrink-lab/bentopy)

| Aşama | Komut | İşlev |
|---|---|---|
| 1 | `bentopy pack` | Yapıların çakışmasız olarak kutuya yerleştirilmesi |
| 2 | `bentopy render` | Yerleşim planından koordinatların üretilmesi |
| 3 | `bentopy solvate` | Kalan boşluğun çözücü ve iyonlarla doldurulması |


---
## 1. Yazılım kurulumu


In [ ]:
%%capture
!pip install -q bentopy
!apt-get -qq update && apt-get -qq install -y gromacs


In [ ]:
!bentopy --help 2>&1 | head -20


---
## 2. Girdi: Oturum 4'te üretilen kaba-taneli model

Oturum 4'te üretilen `at2r_cg.pdb` dosyası kullanılmaktadır. Dosya mevcut
değilse aşağıdaki hücre kurs deposundan hazır olanı indirmektedir.


In [ ]:
import os
if not os.path.exists('at2r_cg.pdb'):
    !git clone -q https://github.com/eygpcr/biyofizik2026-martini.git repo
    !cp repo/04_martini_input/cikti/at2r_cg.pdb . 2>/dev/null || echo 'Hazir dosya bulunamadi'
!ls -la at2r_cg.pdb 2>/dev/null || echo 'Oturum 4 not defterinden at2r_cg.pdb yuklenmelidir'


In [ ]:
# bentopy GRO bicimi beklemektedir
!gmx editconf -f at2r_cg.pdb -o at2r_cg.gro -d 0.5 2>&1 | tail -5


---
## 3. Yerleşim tanımının hazırlanması

Kutu boyutu, yerleştirilecek yapılar ve kopya sayıları bir JSON dosyasıyla
tanımlanmaktadır. `number` değerinin artırılması kalabalık derecesini
artırmakta, buna karşılık paketleme süresini uzatmaktadır.


In [ ]:
import json

yerlesim = {
    'space': {
        'size': [40, 40, 20],        # kutu boyutlari (nm)
        'resolution': 0.5,           # paketleme izgara cozunurlugu (nm)
        'compartments': [
            {'id': 'kutu', 'shape': 'cuboid'}
        ],
    },
    'output': {'title': 'Kalabalik AT2R membrani', 'topol_includes': []},
    'segments': [
        {
            'name': 'AT2R',
            'number': 25,            # kopya sayisi
            'path': 'at2r_cg.gro',
            'compartments': ['kutu'],
        }
    ],
}

json.dump(yerlesim, open('yerlesim.json','w'), indent=2)
print(open('yerlesim.json').read())


---
## 4. `bentopy pack` — yerleşim planının üretilmesi

Bu aşamada koordinat üretilmemekte, yalnızca yapıların konum ve yönelimleri
hesaplanmaktadır.


In [ ]:
!bentopy pack yerlesim.json -o plan.json 2>&1 | tail -20


---
## 5. `bentopy render` — koordinatların üretilmesi


In [ ]:
!bentopy render plan.json -o kalabalik.gro -t kalabalik.top 2>&1 | tail -20

import os
if os.path.exists('kalabalik.gro'):
    n = int(open('kalabalik.gro').read().splitlines()[1])
    print(f'Toplam parcacik sayisi: {n:,}')


---
## 6. `bentopy solvate` — çözücü ve iyon eklenmesi


In [ ]:
!bentopy solvate -f kalabalik.gro -o sistem_kalabalik.gro 2>&1 | tail -20


---
## 7. Ölçek karşılaştırması


In [ ]:
import os
def parcacik(p):
    try:
        return int(open(p).read().splitlines()[1])
    except Exception:
        return None

for ad, dosya in [('Oturum 4 (tek reseptor)', 'sistem.gro'),
                  ('Oturum 6 (kalabalik)', 'sistem_kalabalik.gro')]:
    n = parcacik(dosya) if os.path.exists(dosya) else None
    print(f'{ad:<28}: {n:,} parcacik' if n else f'{ad:<28}: dosya bulunamadi')


---
## Sorun giderme

`bentopy` kurulumunda ortam kaynaklı sorunlar oluşabilmektedir. Oturum için
ayrılan süre 40 dakika olduğundan, sorun yaşanması hâlinde uygulama
sonlandırılarak aşağıdaki materyaller kullanılacaktır:

- Komutlar ve yapılandırma şablonu: [`06_bentopy/kodlar/`](https://github.com/eygpcr/biyofizik2026-martini/tree/main/06_bentopy/kodlar)
- Uygulamanın tam kaydı: [`VIDEO.md`](https://github.com/eygpcr/biyofizik2026-martini/blob/main/06_bentopy/VIDEO.md)
- Önceden üretilmiş çıktı dosyaları

---

## İleri çalışma kaynakları

- [bentopy resmî öğretim materyali](https://cgmartini.nl/docs/tutorials/Martini3/Bentopy/)
- [bentopy deposu ve belgelendirmesi](https://github.com/marrink-lab/bentopy)
- [Protein kompleksleri — Martini](https://cgmartini.nl/docs/tutorials/Martini3/ProteinsIIb/)
- [TS2CG v2.0](https://github.com/weria-pezeshkian/TS2CG-v2.0/wiki/Tutorial)

Ayrıca bkz. [`ILERI_OKUMA.md`](https://github.com/eygpcr/biyofizik2026-martini/blob/main/ILERI_OKUMA.md)
